# RAD Quickstart - LANL

This notebook gives a simple, reproducible way to run LANL RAD-style experiments from the submission folder.

It includes:
- A quick smoke run for RAD six ablations
- Optional supervised RDL baseline run
- Helpers to read output CSVs

In [ ]:
from pathlib import Path
import os

candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd().parent.parent,
    Path('/array/dahlenw/RAD/submission_2026-05-17'),
    Path('/array/dahlenw/RAD'),
]

rad_root = None
for c in candidates:
    if (c / 'lanl' / 'run_lanl_six_ablations.py').exists():
        rad_root = c
        break

if rad_root is None:
    raise FileNotFoundError('Could not locate RAD project root with lanl/run_lanl_six_ablations.py')

print('RAD root:', rad_root)
print('Python   :', os.environ.get('VIRTUAL_ENV', 'system python'))

In [ ]:
import subprocess

RUN_TARGET = 'rad'  # 'rad' or 'rdl'
RUN_NOW = False

env = os.environ.copy()
env.update({
    'START_SEED': '0',
    'N_SEEDS': '1',
})

if RUN_TARGET == 'rad':
    cmd = ['python', str(rad_root / 'lanl' / 'run_lanl_six_ablations.py')]
else:
    env.update({
        'RDL_USE_POS_WEIGHT': '0',
        'RDL_RESULTS_DIR': str(rad_root / 'results' / 'lanl'),
    })
    cmd = ['python', str(rad_root / 'lanl' / 'rdl_lanl.py')]

print('Target :', RUN_TARGET)
print('Command:', ' '.join(cmd))
print('RUN_NOW :', RUN_NOW)

if RUN_NOW:
    p = subprocess.Popen(cmd, cwd=rad_root / 'lanl', env=env)
    rc = p.wait()
    print('Exit code:', rc)
else:
    print('Dry run only. Flip RUN_NOW=True to execute.')

In [ ]:
import pandas as pd

candidate_csvs = [
    rad_root / 'results' / 'lanl' / 'lanl_six_ablations_mean_std.csv',
    rad_root / 'results' / 'lanl' / 'lanl_rdl_unweighted_summary_mean_std.csv',
    rad_root / 'results' / 'lanl' / 'lanl_rdl_weighted_summary_mean_std.csv',
]

for p in candidate_csvs:
    if p.exists():
        print('Loaded:', p)
        display(pd.read_csv(p).head(20))
    else:
        print('Missing:', p)

In [ ]:
# Optional: watch live tmux logs (if runs are active).
log_candidates = [
    rad_root / 'results' / 'lanl' / 'lanl_rdl_unweighted_tmux.log',
    rad_root / 'results' / 'lanl' / 'lanl_rdl_weighted_tmux.log',
]
for p in log_candidates:
    print('Exists' if p.exists() else 'Missing', '-', p)